# Project: DQN from scratch on CartPole

Deep Q-Networks (Mnih et al., 2015) put a neural network in place of the Q-table, made stable by two tricks: a replay buffer and a target network. Build it, solve CartPole-v1 (average return 475+ out of 500), then remove each trick and see what breaks. Training takes about 10-15 minutes (a GPU doesn't help much here; CartPole is tiny).

**How to use this notebook:** run the cells from top to bottom (Shift+Enter). In each step, replace every `___` in the code cell, run it, then run the **Check** cell under it. Stuck? Expand the hidden **Solution** cell (click the ▸ arrow). For speed, turn on a GPU first: *Runtime → Change runtime type → T4 GPU*.

## Setup
Run this once.

In [ ]:
!pip -q install gymnasium
import os, random, collections, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
TEST = os.environ.get("NB_TEST") == "1"          # used only by the automatic tests; ignore it
torch.manual_seed(0); random.seed(0); np.random.seed(0)
env = gym.make("CartPole-v1")
print(env.observation_space, env.action_space)

## Step 1: The Q-network

A small MLP maps the 4 state numbers to one Q-value per action.

In [ ]:
def make_qnet():
    return nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, ___))

qnet = make_qnet()
print(qnet(torch.zeros(1, 4)))

In [ ]:
# Check: run me after the cell above
assert make_qnet()(torch.zeros(3, 4)).shape == (3, 2), "One output per action: 2"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
def make_qnet():
    return nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))

qnet = make_qnet()
print(qnet(torch.zeros(1, 4)))

## Step 2: A replay buffer

Store transitions (s, a, r, s', done) and train on random mini-batches of them. Random sampling breaks the strong correlation between consecutive steps.

In [ ]:
class Replay:
    def __init__(self, cap=50000):
        self.buf = collections.deque(maxlen=cap)
    def push(self, *transition):
        self.buf.append(transition)
    def sample(self, n):
        batch = ___
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (torch.tensor(s, dtype=torch.float32), torch.tensor(a), torch.tensor(r, dtype=torch.float32),
                torch.tensor(s2, dtype=torch.float32), torch.tensor(d, dtype=torch.float32))
    def __len__(self):
        return len(self.buf)

In [ ]:
# Check: run me after the cell above
rb = Replay()
for i in range(100):
    rb.push(np.zeros(4), i % 2, 1.0, np.ones(4), False)
s, a, r, s2, d = rb.sample(32)
assert s.shape == (32, 4) and a.shape == (32,) and len(set(a.tolist())) == 2
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
class Replay:
    def __init__(self, cap=50000):
        self.buf = collections.deque(maxlen=cap)
    def push(self, *transition):
        self.buf.append(transition)
    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (torch.tensor(s, dtype=torch.float32), torch.tensor(a), torch.tensor(r, dtype=torch.float32),
                torch.tensor(s2, dtype=torch.float32), torch.tensor(d, dtype=torch.float32))
    def __len__(self):
        return len(self.buf)

## Step 3: The DQN loss

The target for Q(s, a) is r + gamma max_a' Q_target(s', a'), with no future value when the episode ended. Compute it with no gradient, from the separate target network.

In [ ]:
gamma = 0.99
def dqn_loss(qnet, target_net, batch):
    s, a, r, s2, d = batch
    q = qnet(s).gather(1, a.view(-1, 1)).squeeze(1)
    with torch.no_grad():
        target = ___
    return F.smooth_l1_loss(q, target)

In [ ]:
# Check: run me after the cell above
tn = make_qnet()
batch = (torch.zeros(4, 4), torch.tensor([0, 1, 0, 1]), torch.ones(4), torch.zeros(4, 4), torch.tensor([0., 0., 1., 1.]))
q = qnet(batch[0]).gather(1, batch[1].view(-1, 1)).squeeze(1)
want = 1 + gamma * torch.tensor([1., 1., 0., 0.]) * tn(batch[3]).max(1).values
assert torch.allclose(dqn_loss(qnet, tn, batch), F.smooth_l1_loss(q, want.detach())), "target = r + gamma * (1 - done) * max over next Q-values"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
gamma = 0.99
def dqn_loss(qnet, target_net, batch):
    s, a, r, s2, d = batch
    q = qnet(s).gather(1, a.view(-1, 1)).squeeze(1)
    with torch.no_grad():
        target = r + gamma * (1 - d) * target_net(s2).max(dim=1).values
    return F.smooth_l1_loss(q, target)

## Step 4: Train it

Epsilon decays from 1 to 0.05 over the first 3,000 steps. After 500 warm-up steps, do one gradient step per environment step; copy the online network into the target network every 250 steps. Watch the curve: DQN often reaches 500, then wobbles or even collapses for a while. That instability is real and famous, and it is why the later algorithms (Double DQN, PPO) exist.

In [ ]:
def train_dqn(episodes=300, use_replay=True, target_every=250, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    qnet, target_net = make_qnet(), make_qnet()
    target_net.load_state_dict(qnet.state_dict())
    opt = torch.optim.Adam(qnet.parameters(), lr=1e-3)
    rb, steps, returns = Replay(), 0, []
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep); done, total = False, 0.0
        while not done:
            eps = max(0.05, 1 - steps / 3000)
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(qnet(torch.tensor(s, dtype=torch.float32)).argmax())
            s2, r, term, trunc, _ = env.step(a); done = term or trunc
            rb.push(s, a, r, s2, float(term)); s, total, steps = s2, total + r, steps + 1
            if len(rb) >= 500:
                batch = rb.sample(64) if use_replay else Replay.sample(type("B", (), {"buf": list(rb.buf)[-64:]})(), 64)
                loss = dqn_loss(qnet, target_net, batch)
                opt.zero_grad(); loss.backward(); opt.step()
            if steps % target_every == 0:
                target_net.load_state_dict(___)
        returns.append(total)
        if ep % 25 == 0:
            print(ep, np.mean(returns[-25:]))
    return returns

returns = train_dqn(episodes=5 if TEST else 300)
plt.plot(returns); plt.plot(np.convolve(returns, np.ones(20) / 20, mode="valid")); plt.xlabel("episode"); plt.ylabel("return")

In [ ]:
# Check: run me after the cell above
assert len(returns) > 0
if not TEST:
    assert max(np.convolve(returns, np.ones(20) / 20, mode="valid")) > 400, "A 20-episode average should get above 400 at some point"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
def train_dqn(episodes=300, use_replay=True, target_every=250, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    qnet, target_net = make_qnet(), make_qnet()
    target_net.load_state_dict(qnet.state_dict())
    opt = torch.optim.Adam(qnet.parameters(), lr=1e-3)
    rb, steps, returns = Replay(), 0, []
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep); done, total = False, 0.0
        while not done:
            eps = max(0.05, 1 - steps / 3000)
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(qnet(torch.tensor(s, dtype=torch.float32)).argmax())
            s2, r, term, trunc, _ = env.step(a); done = term or trunc
            rb.push(s, a, r, s2, float(term)); s, total, steps = s2, total + r, steps + 1
            if len(rb) >= 500:
                batch = rb.sample(64) if use_replay else Replay.sample(type("B", (), {"buf": list(rb.buf)[-64:]})(), 64)
                loss = dqn_loss(qnet, target_net, batch)
                opt.zero_grad(); loss.backward(); opt.step()
            if steps % target_every == 0:
                target_net.load_state_dict(qnet.state_dict())
        returns.append(total)
        if ep % 25 == 0:
            print(ep, np.mean(returns[-25:]))
    return returns

returns = train_dqn(episodes=5 if TEST else 300)
plt.plot(returns); plt.plot(np.convolve(returns, np.ones(20) / 20, mode="valid")); plt.xlabel("episode"); plt.ylabel("return")

## Step 5: Ablation: take the tricks away

Run three more agents: no replay buffer (train only on the latest transitions), no target network (copy every step), and neither. Compare learning curves. This is the experiment from the DQN paper; expect the stripped-down agents to be slower and much less stable. (Takes 10-20 minutes.)

In [ ]:
variants = {
    "full DQN": dict(),
    "no replay": dict(use_replay=False),
    "no target net": dict(target_every=1),
    "neither": dict(use_replay=False, target_every=1),
}
curves = {name: train_dqn(episodes=3 if TEST else 300, **kw) for name, kw in variants.items()}
for name, r in curves.items():
    plt.plot(np.convolve(r, np.ones(min(20, len(r))) / min(20, len(r)), mode="valid"), label=name)
plt.xlabel("episode"); plt.ylabel("return (20-episode average)"); plt.legend()

In [ ]:
# Check: run me after the cell above
assert set(curves) == set(variants)
print("Done! Write two sentences in your repo on what each trick buys you.")

In [ ]:
#@title Solution (click ▸ to show)
variants = {
    "full DQN": dict(),
    "no replay": dict(use_replay=False),
    "no target net": dict(target_every=1),
    "neither": dict(use_replay=False, target_every=1),
}
curves = {name: train_dqn(episodes=3 if TEST else 300, **kw) for name, kw in variants.items()}
for name, r in curves.items():
    plt.plot(np.convolve(r, np.ones(min(20, len(r))) / min(20, len(r)), mode="valid"), label=name)
plt.xlabel("episode"); plt.ylabel("return (20-episode average)"); plt.legend()